In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

<font size="5" color="black">ch3. 연관 분석</font>
- pip install apyori

# 1. 연관분석 개요
- 데이터 간의 서로 다른 사건이나 거래에서 함께 발생하는 규칙과 관계를 발견하는 데이터 마이닝 기법
- 대표 활용 사례 : 유통/커머스(상품 진열 위치 최적화, 교차 판매 추천), 금융/보험(보험 사기 적발, 이상 거래 감지) 등
<br>
- 조건(left-hand side, item_base) ; 오렌지 주스($x$) → 결과(right-hand side, item_add) ; 와인($y$)

- 연관분석 지표
    - 지지도 : 전체 거래 중 항목 A와 B가 동시에 포함된 거래의 비율
        - $\text{support}(A → B) = \frac{P(A \cap B)}{\text{전체 거래 수}}$ = 0.2
    - 신뢰도 : 항목 A가 구매된 거래 중 항목 B도 함께 구매된 비율 (조건부 확률)
        - $\text{confidence}(A → B) = \frac{P(A \cap B)}{P(A)}$ = 0.5
    - 향상도 : A의 구매가 B의 구매 확률을 얼마나 증가시키는지를 나타내는 비율
        - $\text{lift}(A → B) = \frac{P(A \cap B)}{P(A) \times P(B)}$ = 0.833
            * $\text{Lift} > 1$ : 양의 연관성 (A와 B를 함께 구매할 확률이 상호 독립일 때보다 높음)
            * $\text{Lift} = 1$ : 상호 독립 (연관성 없음)
            * $\text{Lift} < 1$ : 음의 연관성 (상호 독립일 때보다 A와 B를 함께 구매할 확률이 낮음)
<br><br><br>



# 2. 연관분석 구현
- apiori : 한 항목집합이 빈발하다면, 그 항목집합의 모든 부분집합도 빈발하다"는 원리(Apriori Principle)를 이용


In [3]:
import csv
with open('data/cf_basket.csv', encoding='utf-8') as f:
    csvdata = csv.reader(f)
    transaction = list(csvdata)

In [5]:
from apyori import apriori
rules = apriori(transaction,
                min_support=0.15, # 지지도가 15% 이상인 데이터
                min_confidence=0.1, # 신뢰도가 10% 이상인 데이터
                min_lift=1.00001) # 향상도가 1보다 큰 데이터

In [8]:
# 제너레이터 타입 → 리스트
rules = list(rules)

In [9]:
# 리스트 내 항목 확인
rule = rules[5]

In [27]:
# 항목 내의 세부 항복 확인
support = rule[1]
ordered_st = rule[2]
for item in ordered_st:
    lhs = item[0]
    lhs = ', '.join([x for x in lhs])
    rhs = item[1]
    rhs = ', '.join([y for y in rhs])
    confidence = item[2]
    lift = item[3]
    print(f'{lhs} → {rhs} |  confidence : {item[2]} \t lift : {item[3]}')

콜라 → 소주, 와인 |  confidence : 0.25 	 lift : 1.25
소주, 와인 → 콜라 |  confidence : 1.0 	 lift : 1.25


In [36]:
# 항목 및 세부 항복 모두 리스트로 변환 
rules_lst = []
for rule in rules:
    support = rule[1]
    ordered_st = rule[2]
    for item in ordered_st:
        lhs = item[0]
        lhs = ', '.join([x for x in lhs])
        rhs = item[1]
        rhs = ', '.join([y for y in rhs])
        confidence = item[2]
        lift = item[3]
        rules_lst.append({'lhs':lhs,
                          'rhs':rhs,
                          'support':support,
                          'confidence':round(confidence, 2),
                          'lift':round(lift, 2)})

In [38]:
# 데이터 프레임 변환
import pandas as pd
pd.DataFrame(rules_lst, columns=['lhs','rhs','support','confidence','lift'])

,lhs,rhs,support,confidence,lift
0,맥주,콜라,0.4,1.00,1.25
1,콜라,맥주,0.4,0.50,1.25
2,소주,콜라,0.6,1.00,1.25
3,콜라,소주,0.6,0.75,1.25
4,콜라,"맥주, 소주",0.2,0.25,1.25
5,"맥주, 소주",콜라,0.2,1.00,1.25
6,맥주,"콜라, 와인",0.2,0.50,1.25
7,콜라,"맥주, 와인",0.2,0.25,1.25
8,"맥주, 와인",콜라,0.2,1.00,1.25
9,"콜라, 와인",맥주,0.2,0.50,1.25


# 3. 예시 1 : 뉴스 연관분석
- 뉴스 20개 내용의 명사를 추출하여 연관분석 시행

In [41]:
from dotenv import load_dotenv
import os
import requests
import json
import pandas as pd
load_dotenv('C:/ai/source/01_python/.env')

True

In [64]:
client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')

url = f"https://openapi.naver.com/v1/search/news.json"
params = {'query':'인스파이어아레나', 'display':50 ,'sort':'date'}
headers = {
    'X-Naver-Client-Id': client_id,
    'X-Naver-Client-Secret':client_secret
}
response = requests.get(url, params=params, headers=headers)

items = response.json()['items']

# 제목(title) + ' ' + 요약(description) 텍스만 추출하여 리스트 형태로 출력
news_texts = []
for item in items:
    title = item.get('title').replace('<b>', ' ').replace('</b>', ' ')
    description = item.get('description').replace('<b>', ' ').replace('</b>', ' ')
    news_texts.append(title + ' ' + description)
news_texts[:3], len(news_texts)

(['라이즈, ‘키윙’ 그룹 부문 1위 오는 12일과 13일에는 인천  인스파이어 아레나 에서 데뷔 3주년 기념 팬미팅 ‘Ch. RIIZE : ON AIR’를 개최하고 팬들과 만날 예정이다. 한편 그룹 부문 1위를 차지한 라이즈의 소식은 광화문 초대형 디지털 사이니지... ',
  "아일릿 민주, 이효리 '10 Minutes' R&amp;B 커버…몽환충 Y2K 감성 한편민주가 속한 아일릿은 서울을 시작으로 일본 5개 도시와 홍콩을 아우른 아시아 투어를 성황리에 마무리한 가운데, 오는 10월 10~11일, 17~18일 2주간 인천  인스파이어 아레나 에서 'ILLIT LIVE 'PRESS START♥︎' ENCORE in... ",
  "뮤지컬 '드라큘라' 1위 유지…'광화문연가' 5계단 올라 3위 [9월 6일 공연... 이승철 40주년 콘서트, THE VOICE: LEE SEUNG CHUL [인천] - 대중음악 - 2026.12.12 - 인천 -  인스파이어  엔터테인먼트 리조트  아레나  13. 헬스키친 - 뮤지컬 - 2026.07.24 ~ 2026.11.08 - 서울 - GS아트센터 14.... "],
 50)

In [65]:
from konlpy.tag import Hannanum, Kkma, Komoran, Okt
from mecab import MeCab

analyzer = MeCab()

stopwords={'에서'}
news = []

for article in news_texts:
    noun_list = analyzer.nouns(article)
    noun_list = [word for word, tag in analyzer.pos(article) \
                 if tag in ('NNG','NNP') and
                    word not in stopwords and
                    len(word)>1]
    news.append(noun_list)
    
print(news[:3])

[['라이즈', '그룹', '부문', '인천', '인스파이어', '아레나', '데뷔', '기념', '미팅', '개최', '예정', '한편', '그룹', '부문', '차지', '라이즈', '소식', '광화문', '초대', '디지털', '사이', '니지'], ['아일릿', '민주', '이효리', '커버', '환충', '감성', '한편', '민주', '아일릿', '서울', '시작', '일본', '도시', '홍콩', '아시아', '투어', '성황리', '마무리', '가운데', '인천', '인스파이어', '아레나'], ['뮤지컬', '드라큘라', '유지', '광화문', '연가', '계단', '공연', '이승철', '콘서트', '인천', '대중', '음악', '인천', '인스파이어', '엔터', '리조트', '아레나', '헬스', '키친', '뮤지컬', '서울', '아트', '센터']]


In [71]:
rules = apriori(news,
                min_support=0.3,
                min_confidence=0.2,
                min_lift=1.0001)
rules = list(rules)
len(rules)

128

In [72]:
rules_lst = []
for rule in rules:
    support = rule[1]
    ordered_st = rule[2]
    for item in ordered_st:
        lhs = item[0]
        lhs = ', '.join([x for x in lhs])
        rhs = item[1]
        rhs = ', '.join([y for y in rhs])
        confidence = item[2]
        lift = item[3]
        rules_lst.append({'lhs':lhs,
                          'rhs':rhs,
                          'support':support,
                          'confidence':round(confidence, 2),
                          'lift':round(lift, 2)})

In [73]:
df = pd.DataFrame(rules_lst, columns=['lhs','rhs','support','confidence','lift'])

In [74]:
df.sort_values(by=['lift', 'confidence', 'support'], ascending=False, inplace=True)
df

,lhs,rhs,support,confidence,lift
4,기념,라이즈,0.32,1.00,2.94
10,데뷔,라이즈,0.32,1.00,2.94
64,기념,"라이즈, 미팅",0.32,1.00,2.94
68,"미팅, 기념",라이즈,0.32,1.00,2.94
70,기념,"라이즈, 아레나",0.32,1.00,2.94
...,...,...,...,...,...
1513,"인스파이어, 아레나, 인천","미팅, 기념, 데뷔",0.30,0.31,1.02
1648,인천,"미팅, 기념, 인스파이어, 라이즈, 아레나, 데뷔",0.30,0.31,1.02
1667,"아레나, 인천","미팅, 기념, 인스파이어, 라이즈, 데뷔",0.30,0.31,1.02
1668,"인스파이어, 인천","미팅, 기념, 라이즈, 아레나, 데뷔",0.30,0.31,1.02
